In [5]:
!nvidia-smi

Mon Mar 16 22:12:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:01:00.0  On |                  Off |
|  0%   39C    P8             14W /  450W |    1061MiB /  24564MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%pip install numba
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
from numba import cuda
import numpy as np
print(cuda.gpus)

# Define the cuda kernel
@cuda.jit
def add_kernel(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[idx] = a[idx] + b[idx]

# Host Code (CPU)
n = 1000000
a_host = np.random.randn(n).astype(np.float32)
b_host = np.random.randn(n).astype(np.float32)
c_host = np.empty_like(a_host)

# Allocate memory on the GPU
a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

# Configure the kernel launch parameters of the grid and block
threads_per_block = 256
# Calculate how many blocks we need to cover the entire array
blocks_per_grid = (a_device.size + (threads_per_block - 1)) // threads_per_block
print(f"Launching {blocks_per_grid} blocks of {threads_per_block} threads each")

# Launch the kernel on the gpu
add_kernel[blocks_per_grid, threads_per_block](a_device, b_device, c_device)

# Copy the result back to the host
c_device.copy_to_host(c_host)

if np.allclose(c_host, a_host + b_host):
    print("GPU results match CPU results!")
else:
    print("GPU results do not match CPU results!")
    
    


<Managed Device 0>
Launching 3907 blocks of 256 threads each
GPU results match CPU results!


Modify the code so that instead of writing to c[idx], you write to c[0]

In [8]:
from numba import cuda
import numpy as np
print(cuda.gpus)

# Define the cuda kernel
@cuda.jit
def add_kernel(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[0] = a[idx] + b[idx]

# Host Code (CPU)
n = 1000000
a_host = np.random.randn(n).astype(np.float32)
b_host = np.random.randn(n).astype(np.float32)
c_host = np.empty_like(a_host)

# Allocate memory on the GPU
a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

# Configure the kernel launch parameters of the grid and block
threads_per_block = 256
# Calculate how many blocks we need to cover the entire array
blocks_per_grid = (a_device.size + (threads_per_block - 1)) // threads_per_block
print(f"Launching {blocks_per_grid} blocks of {threads_per_block} threads each")

# Launch the kernel on the gpu
add_kernel[blocks_per_grid, threads_per_block](a_device, b_device, c_device)

# Copy the result back to the host
c_device.copy_to_host(c_host)

if np.allclose(c_host, a_host + b_host):
    print("GPU results match CPU results!")
else:
    print("GPU results do not match CPU results!")

print("c_host[0]: ", c_host[0])
print("c_host[1]: ", c_host[1])
print("a_host[-1] + b_host[-1]: ", a_host[-1] + b_host[-1])

<Managed Device 0>
Launching 3907 blocks of 256 threads each
GPU results do not match CPU results!
c_host[0]:  0.62425584
c_host[1]:  0.0
a_host[-1] + b_host[-1]:  -1.9933991


c_host[0] is some number equal to a_host[-1] + b_host[-1] because it rewrites the same exact index in every thread.

In [9]:
from numba import cuda
import numpy as np
print(cuda.gpus)

# Define the cuda kernel
@cuda.jit
def add_kernel(a, b, c):
    idx = cuda.grid(1)
    # if idx < a.size:
    c[idx] = a[idx] + b[idx]

# Host Code (CPU)
n = 1000000
a_host = np.random.randn(n).astype(np.float32)
b_host = np.random.randn(n).astype(np.float32)
c_host = np.empty_like(a_host)

# Allocate memory on the GPU
a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

# Configure the kernel launch parameters of the grid and block
threads_per_block = 256
# Calculate how many blocks we need to cover the entire array
blocks_per_grid = (a_device.size + (threads_per_block - 1)) // threads_per_block
print(f"Launching {blocks_per_grid} blocks of {threads_per_block} threads each")

# Launch the kernel on the gpu
%timeit add_kernel[blocks_per_grid, threads_per_block](a_device, b_device, c_device)
%timeit c=a_host+b_host

# Copy the result back to the host
c_device.copy_to_host(c_host)

if np.allclose(c_host, a_host + b_host):
    print("GPU results match CPU results!")
else:
    print("GPU results do not match CPU results!")
    
    


<Managed Device 0>
Launching 3907 blocks of 256 threads each
16.4 μs ± 92.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
199 μs ± 1.74 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
GPU results match CPU results!


There is no error when commenting out "if idx < a.size:". However, I think the issue would be that without that check it might run into memory not allocated? I don't know.

%timeit shows that runbning with cuda is more than 10x faster, at ~17ms compared to ~200ms per loop

**PART 2**

In [10]:
!pip install -qqq git+https://github.com/danoneata/chalk@srush-patch-1
!wget -q https://github.com/srush/GPU-Puzzles/raw/main/robot.png https://github.com/srush/GPU-Puzzles/raw/main/lib.py
import numba
import numpy as np
import warnings
from numba import cuda
from lib import CudaProblem, Coord
warnings.filterwarnings ('ignore', module='numba')
print("Setup complete!")

Setup complete!


In [11]:
def map_spec(a):
    return a + 10

def map_test(cuda):
    def call(out, a) -> None:
        local_i = cuda.threadIdx.x
        # FILL ME IN ( roughly 1 line )
        out[local_i] = a[local_i] + 10

    return call

SIZE = 4
out = np.zeros((SIZE,))
a = np.arange(SIZE)
problem = CudaProblem(
    "Map", map_test, [a], out, threadsperblock=Coord(SIZE, 1), spec=map_spec
)
problem.check()

Passed Tests!


In [12]:
def zip_spec(a , b):
    return a + b

def zip_test(cuda):
    def call(out, a, b) -> None :
        local_i = cuda.threadIdx.x
        # FILL ME IN ( roughly 1 lines )
        out[local_i] = a[local_i] + b[local_i]

    return call

SIZE = 4
out = np.zeros((SIZE,))
a = np.arange(SIZE)
b = np.arange(SIZE)
problem = CudaProblem(
    "Zip", zip_test, [a,b], out, threadsperblock=Coord(SIZE, 1), spec=zip_spec
)
problem.check()

Passed Tests!


In [13]:
def map_guard_test(cuda):
    def call(out, a, size) -> None:
        local_i = cuda.threadIdx.x
        # FILL ME IN ( roughly 2 lines )
        if local_i < size:
            out[local_i] = a[local_i] + 10


    return call

SIZE = 4
out = np.zeros((SIZE,))
a = np.arange(SIZE)
problem = CudaProblem(
    "Guard",
    map_guard_test,
    [a],
    out,
    [SIZE],
    threadsperblock = Coord(8, 1),
    spec=map_spec,
)
problem.check()

Passed Tests!


In [15]:
def map_2D_test(cuda):
    def call(out, a, size) -> None:
        local_i = cuda.threadIdx.x
        local_j = cuda.threadIdx.y
        # FILL ME IN ( roughly 2 lines )
        if local_i<size and local_j<size:
            out[local_i, local_j] = a[local_i, local_j] + 10
        

    return call

SIZE = 2
out = np.zeros((SIZE, SIZE))
a = np.arange(SIZE * SIZE).reshape((SIZE, SIZE))
problem = CudaProblem(
    " Map 2D", map_2D_test, [a], out, [SIZE], threadsperblock=Coord(3, 3), spec=map_spec
)
problem.check()

Passed Tests!


In [18]:
def map_block_test(cuda):
    def call(out, a, size) -> None:
        i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        # FILL ME IN ( roughly 2 lines )
        if i < size:
            out[i] = a[i] + 10

    
    return call

SIZE = 9
out = np.zeros((SIZE,))
a = np.arange(SIZE)
problem = CudaProblem(
    "Blocks",
    map_block_test,
    [a],
    out,
    [SIZE],
    threadsperblock=Coord(4, 1),
    blockspergrid=Coord(3, 1),
    spec=map_spec,
)
problem.check()

Passed Tests!


In [19]:
def map_block2D_test(cuda):
    def call(out, a, size) -> None:
        i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        # FILL ME IN : calculate the ’j ’ index and perform the operation ( roughly 3 lines )
        j = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
        if i<size and j<size:
            out[i, j] = a[i, j] + 10
    
    return call

SIZE = 5
out = np.zeros((SIZE, SIZE))
a = np.ones((SIZE, SIZE))
problem = CudaProblem(
    "Blocks 2D",
    map_block2D_test,
    [a],
    out,
    [SIZE],
    threadsperblock=Coord(3, 3),
    blockspergrid=Coord(2, 2),
    spec = map_spec,
)
problem.show()
problem.check()

# Blocks 2D
 
   Score (Max Per Thread):
   |  Global Reads | Global Writes |  Shared Reads | Shared Writes |
   |             1 |             1 |             0 |             0 | 

Passed Tests!
